In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

demo_rows = [
    ('I was charged twice for my subscription this month and need a refund.', 'Billing', 'High'),
    ('My invoice is missing from the portal and I cannot download it.', 'Billing', 'Medium'),
    ('Please update my payment method because my credit card expired.', 'Billing', 'Medium'),
    ('The refund for my canceled plan has not arrived yet.', 'Billing', 'High'),
    ('I want to change my billing address for future invoices.', 'Billing', 'Low'),
    ('My account was locked after several failed login attempts.', 'Account', 'High'),
    ('I need to reset my password because I forgot it.', 'Account', 'Medium'),
    ('Please change the email address linked to my account.', 'Account', 'Low'),
    ('Two-factor authentication is not working on my phone.', 'Account', 'High'),
    ('I would like to delete my profile and close the account.', 'Account', 'Medium'),
    ('The app crashes every time I open the dashboard.', 'Technical Issue', 'High'),
    ('The website is running very slowly during checkout.', 'Technical Issue', 'High'),
    ('An error appears when I try to upload a file.', 'Technical Issue', 'Medium'),
    ('The search filter does not return the correct results.', 'Technical Issue', 'Medium'),
    ('Notifications are delayed and sometimes never arrive.', 'Technical Issue', 'Low'),
    ('How do I change my plan from monthly to annual billing?', 'General Query', 'Low'),
    ('Where can I find the user guide for the new dashboard?', 'General Query', 'Low'),
    ('Do you support multiple languages in the product?', 'General Query', 'Low'),
    ('What are the differences between the starter and pro plans?', 'General Query', 'Low'),
    ('Can someone explain how to export my reports?', 'General Query', 'Low'),
    ('My subscription was canceled without warning and I cannot access the service.', 'Billing', 'High'),
    ('I keep getting a server timeout when submitting the form.', 'Technical Issue', 'High'),
    ('I cannot sign in because the verification code never arrives.', 'Account', 'High'),
    ('Please help me understand the latest pricing changes.', 'Billing', 'Low'),
    ('The dashboard layout looks broken on mobile devices.', 'Technical Issue', 'Medium'),
    ('I need help finding my API key in the account settings.', 'General Query', 'Low'),
    ('Please remove my old card and keep the new one as default payment.', 'Billing', 'Medium'),
    ('The order confirmation page shows a blank screen after payment.', 'Technical Issue', 'High'),
    ('I accidentally created duplicate accounts with the same email.', 'Account', 'Medium'),
    ('Can I switch my workspace owner to another team member?', 'General Query', 'Low'),
]

tickets = pd.DataFrame(demo_rows, columns=['ticket_text', 'category', 'priority'])
tickets.head()

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [token for token in text.split() if token not in ENGLISH_STOP_WORDS]
    return ' '.join(tokens)

tickets['clean_text'] = tickets['ticket_text'].apply(clean_text)
tickets[['ticket_text', 'clean_text', 'category', 'priority']].head(10)

In [ ]:
X = tickets['clean_text']
y_category = tickets['category']

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X, y_category, test_size=0.25, random_state=42, stratify=y_category
)

category_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

category_model.fit(X_train_cat, y_train_cat)
category_predictions = category_model.predict(X_test_cat)

print('Category Accuracy:', round(accuracy_score(y_test_cat, category_predictions), 3))
print('Category Precision (macro):', round(precision_score(y_test_cat, category_predictions, average='macro', zero_division=0), 3))
print('Category Recall (macro):', round(recall_score(y_test_cat, category_predictions, average='macro', zero_division=0), 3))
print()
print(classification_report(y_test_cat, category_predictions, zero_division=0))

In [ ]:
category_labels = sorted(tickets['category'].unique())
cm_category = confusion_matrix(y_test_cat, category_predictions, labels=category_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_category, display_labels=category_labels)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
ax.set_title('Ticket Category Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
y_priority = tickets['priority']

X_train_pri, X_test_pri, y_train_pri, y_test_pri = train_test_split(
    X, y_priority, test_size=0.25, random_state=42, stratify=y_priority
)

priority_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

priority_model.fit(X_train_pri, y_train_pri)
priority_predictions = priority_model.predict(X_test_pri)

print('Priority Accuracy:', round(accuracy_score(y_test_pri, priority_predictions), 3))
print('Priority Precision (macro):', round(precision_score(y_test_pri, priority_predictions, average='macro', zero_division=0), 3))
print('Priority Recall (macro):', round(recall_score(y_test_pri, priority_predictions, average='macro', zero_division=0), 3))
print()
print(classification_report(y_test_pri, priority_predictions, zero_division=0))

In [ ]:
priority_labels = sorted(tickets['priority'].unique())
cm_priority = confusion_matrix(y_test_pri, priority_predictions, labels=priority_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_priority, display_labels=priority_labels)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Greens', values_format='d', colorbar=False)
ax.set_title('Priority Confusion Matrix')
plt.tight_layout()
plt.show()

def predict_ticket(ticket_text: str):
    cleaned = clean_text(ticket_text)
    predicted_category = category_model.predict([cleaned])[0]
    predicted_priority = priority_model.predict([cleaned])[0]
    return {'cleaned_text': cleaned, 'category': predicted_category, 'priority': predicted_priority}

sample_ticket = 'I was charged twice and my account is locked after a failed login attempt.'
predict_ticket(sample_ticket)

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42
STOP_WORDS = set(ENGLISH_STOP_WORDS)

In [ ]:
def infer_column(columns, keywords):
    normalized = {column: column.lower() for column in columns}
    for keyword in keywords:
        for column, lower_name in normalized.items():
            if keyword in lower_name:
                return column
    return None

def assign_priority(text):
    text = str(text).lower()
    high_terms = ["outage", "down", "breach", "compromised", "locked out", "urgent", "cannot access", "data loss", "security"]
    medium_terms = ["failed", "error", "slow", "issue", "bug", "stuck", "problem", "double charge", "verify", "reset"]
    if any(term in text for term in high_terms):
        return "High"
    if any(term in text for term in medium_terms):
        return "Medium"
    return "Low"

def build_demo_data():
    ticket_bank = {
        "Billing": {
            "High": [
                "My account was charged twice and I need this fixed immediately.",
                "The payment failed but my card was still debited. Please investigate now.",
                "I was incorrectly billed after canceling, and the subscription should be stopped today."
            ],
            "Medium": [
                "I need help understanding an unexpected invoice fee.",
                "Please update the billing address on my account.",
                "My refund has not appeared yet and I would like an update."
            ],
            "Low": [
                "Can you send me a copy of my last invoice?",
                "I want to change my payment method before the next renewal.",
                "What is the difference between the monthly and annual plans?"
            ]
        },
        "Technical Issue": {
            "High": [
                "The app is down for everyone in my team and we cannot log in.",
                "A security warning appeared after login and I need help immediately.",
                "We are losing data during sync and this is blocking production work."
            ],
            "Medium": [
                "The dashboard is loading very slowly on Chrome.",
                "An error appears when I export the report.",
                "The integration with Slack stopped syncing messages."
            ],
            "Low": [
                "How do I change the theme in the app settings?",
                "Can you explain how to use the new filter option?",
                "I would like to request a feature for dark mode scheduling."
            ]
        },
        "Account": {
            "High": [
                "My account is locked and I cannot access it after multiple login attempts.",
                "I think my account was compromised and I need it secured right away.",
                "Someone changed my email address and I no longer have control of the account."
            ],
            "Medium": [
                "I am not receiving the verification code for my login.",
                "Please help me reset my password because the reset link expired.",
                "Two-factor authentication is failing on my device."
            ],
            "Low": [
                "How do I update my profile picture?",
                "Please change the display name on my account.",
                "Where can I manage notification preferences?"
            ]
        },
        "General Query": {
            "High": [
                "I am escalating a serious complaint about the service experience.",
                "This issue affects our operations and needs immediate attention.",
                "We need urgent clarification because the current process is blocking our launch."
            ],
            "Medium": [
                "Can you clarify the API rate limits for my plan?",
                "I need help understanding the onboarding steps.",
                "Please explain how your SSO setup works."
            ],
            "Low": [
                "I want to know more about the product roadmap.",
                "Can someone share pricing details for enterprise features?",
                "Where can I find the documentation for your platform?"
            ]
        }
    }

    records = []
    for category, priority_groups in ticket_bank.items():
        for priority, texts in priority_groups.items():
            for text in texts:
                records.append({
                    "ticket_text": text,
                    "category": category,
                    "priority": priority
                })

    return pd.DataFrame(records)

def load_support_ticket_data():
    candidates = [
        "support_tickets.csv",
        "customer_support_tickets.csv",
        "tickets.csv",
        "it_support_tickets.csv",
        "support_ticket_dataset.csv"
    ]

    for name in candidates:
        path = Path(name)
        if not path.exists():
            continue

        data = pd.read_csv(path)
        text_column = infer_column(data.columns, ["ticket_text", "text", "description", "issue", "message", "summary"])
        category_column = infer_column(data.columns, ["category", "topic", "type", "label"])
        priority_column = infer_column(data.columns, ["priority", "severity", "urgency"])

        if text_column is None or category_column is None:
            continue

        cleaned = data[[text_column, category_column]].copy()
        cleaned.columns = ["ticket_text", "category"]
        cleaned["ticket_text"] = cleaned["ticket_text"].astype(str)
        cleaned["category"] = cleaned["category"].astype(str)

        if priority_column is not None:
            cleaned["priority"] = data[priority_column].astype(str)
        else:
            cleaned["priority"] = cleaned["ticket_text"].apply(assign_priority)

        return cleaned.dropna().reset_index(drop=True), path

    return build_demo_data(), None

tickets_df, source_path = load_support_ticket_data()
print("Data source:", source_path if source_path else "demo dataset created in notebook")
print("Shape:", tickets_df.shape)
tickets_df.head()

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    tokens = [token for token in text.split() if token not in STOP_WORDS and len(token) > 2]
    return " \
,
,
clean_text"] = tickets_df["ticket_text"].apply(clean_text)
tickets_df[["ticket_text", "clean_text", "category", "priority"]].head(10)
execution_count
outputs

In [ ]:
print("Category distribution:")
display(tickets_df["category"].value_counts())

print("Priority distribution:")
display(tickets_df["priority"].value_counts())

In [ ]:
def train_and_evaluate_text_model(dataframe, target_column, title):
    X = dataframe["clean_text"]
    y = dataframe[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=2000))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print(f"{title} accuracy: {accuracy_score(y_test, predictions):.3f}")
    print()
    print(classification_report(y_test, predictions))

    labels = sorted(y.unique())
    matrix = confusion_matrix(y_test, predictions, labels=labels)

    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"{title} confusion matrix")

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, matrix[i, j], ha="center", va="center", color="black")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    return model

category_model = train_and_evaluate_text_model(tickets_df, "category", "Category classification")

In [ ]:
priority_model = train_and_evaluate_text_model(tickets_df, "priority", "Priority classification")

In [ ]:
sample_tickets = pd.DataFrame({
    "ticket_text": [
        "I cannot log in after resetting my password and my account is locked.",
        "Can you explain why I was charged twice this month?",
        "The dashboard is extremely slow and export keeps failing.",
        "Please share the pricing details for your enterprise plan."
    ]
})
sample_tickets["clean_text"] = sample_tickets["ticket_text"].apply(clean_text)
sample_tickets["predicted_category"] = category_model.predict(sample_tickets["clean_text"])
sample_tickets["predicted_priority"] = priority_model.predict(sample_tickets["clean_text"])
sample_tickets